Random Forest – Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

# Parameter grid for Random Forest
rf_params = {
    "n_estimators": [100, 200, 500],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42)

# Grid Search
rf_grid = GridSearchCV(rf, rf_params, cv=5, scoring="roc_auc", n_jobs=-1, verbose=1)
rf_grid.fit(X_reduced, y_reduced)

print("Best Random Forest Params:", rf_grid.best_params_)
print("Best Random Forest AUC:", rf_grid.best_score_)


Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Random Forest Params: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100}
Best Random Forest AUC: 0.901140873015873


SVM (RBF) – Hyperparameter Tuning

In [ ]:
from sklearn.svm import SVC

# Parameter grid for SVM
svm_params = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.1, 0.01, 0.001],
    "kernel": ["rbf"]
}

svm = SVC(probability=True, random_state=42)

# Randomized Search (faster than full grid for SVM)
svm_random = RandomizedSearchCV(svm, svm_params, cv=5, scoring="roc_auc", n_jobs=-1, n_iter=8, random_state=42, verbose=1)
svm_random.fit(X_reduced, y_reduced)

print("Best SVM Params:", svm_random.best_params_)
print("Best SVM AUC:", svm_random.best_score_)


Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best SVM Params: {'kernel': 'rbf', 'gamma': 0.01, 'C': 100}
Best SVM AUC: 0.9083085317460318


In [ ]:
pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from joblib import dump

# Build pipeline with scaling + best SVM
best_svm = SVC(
    kernel="rbf", 
    C=1, 
    gamma="scale", 
    probability=True, 
    random_state=42
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),  # scale numeric features
    ("clf", best_svm)              # classifier
])

# Train final pipeline on the full dataset
pipeline.fit(X_reduced, y_reduced)


Pipeline(steps=[('scaler', StandardScaler()),
                ('clf', SVC(C=1, probability=True, random_state=42))])

In [ ]:
# Save pipeline as .pkl
dump(pipeline, "final_model.pkl")
print("✅ Model saved as final_model.pkl")


✅ Model saved as final_model.pkl
